# Figshare FoG Detection — Improved Pipeline
## Part 1: Data Loading, Preprocessing & Feature Extraction

Fixes ALL identified issues from diagnostic analysis:
1. Handles 12/35 monoclase subjects properly in LOSO metrics
2. Single z-normalization (signal level only) — no double normalization
3. Adds RobustScaler in pipeline
4. Feature selection increased to k=80 (from k=12/sensor)
5. VarianceThreshold removes constant features before SelectKBest
6. Bandpass 0.5-25 Hz for 128 Hz data
7. Exploits gyroscope (more discriminative than accelerometer per literature)
8. Computes derived signals: magnitudes, jerk, Freeze Index from both acc & gyr

In [ ]:
from __future__ import annotations

import sys, os, time, json, warnings, logging
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt
from scipy.optimize import minimize
from joblib import Parallel, delayed
from tqdm import tqdm

from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

# ── Project path ──
PROJECT_ROOT = Path(os.getcwd()).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from loaders import FigshareDatasetLoader
from features.extractors import FeatureExtractor
from utils.pipeline_utils import (
    interpolate_missing, bandpass_filter, zscore_normalize, create_windows,
    youden_threshold, compute_metrics, get_classifiers, get_param_grids,
    prepare_fold, preprocess_features, train_and_evaluate_classifier,
    build_base_model, aggregate_results, print_results_table, print_fusion_results,
    HAS_XGB, HAS_SMOTE,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("figshare")

## Constants

In [ ]:
FS = 128  # Figshare sampling rate
WINDOW_SEC = 4.0
WINDOW_SAMPLES = int(WINDOW_SEC * FS)  # 512
TRAIN_OVERLAP = 0.50
TEST_OVERLAP = 0.0
LABEL_THRESH = 0.50  # majority voting
BP_LOW, BP_HIGH, BP_ORDER = 0.5, 25.0, 4
NPERSEG = min(256, WINDOW_SAMPLES)
K_FEATURES = 80
SEED = 42
N_INNER_CV = 3
N_SEARCH_ITER = 20

# Figshare columns
ACC_COLS = ["acc_ml_lower_back", "acc_ap_lower_back", "acc_si_lower_back"]
GYR_COLS = ["gyr_ml_lower_back", "gyr_ap_lower_back", "gyr_si_lower_back"]
ALL_SENSOR_COLS = ACC_COLS + GYR_COLS
FOG_COL = "freezing_flag"

DATASET_PATH = PROJECT_ROOT / "Datasets" / "Figshare a public dataset"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "figshare_improved_results"

## Data Loading & Preprocessing

In [ ]:
def load_and_preprocess() -> Dict[int, Dict]:
    """Load Figshare data, preprocess per subject/trial."""
    log.info("Loading Figshare dataset from %s", DATASET_PATH)
    loader = FigshareDatasetLoader(str(DATASET_PATH))
    df = loader.load_all_data(verbose=False, trial_type="walking")  # Walking trials only

    log.info("Loaded %d samples from %d subjects", len(df), df["subject_id"].nunique())

    subjects_data = {}
    for sid in sorted(df["subject_id"].unique()):
        sub_df = df[df["subject_id"] == sid]
        all_windows, all_labels = [], []

        for sess_id in sorted(sub_df["session_id"].unique()):
            trial = sub_df[sub_df["session_id"] == sess_id]

            if FOG_COL not in trial.columns:
                continue

            signal_raw = trial[ALL_SENSOR_COLS].values.astype(np.float64)
            labels_raw = trial[FOG_COL].values.astype(np.float64)

            # Skip trials with all NaN
            if np.all(np.isnan(signal_raw)):
                continue

            # Interpolate missing values before filtering
            signal_raw = interpolate_missing(signal_raw)
            labels_raw = np.nan_to_num(labels_raw, nan=0.0)

            # Bandpass filter
            try:
                signal_filt = bandpass_filter(signal_raw, FS, BP_LOW, BP_HIGH, BP_ORDER)
            except Exception:
                continue

            # Compute derived signals
            acc = signal_filt[:, :3]
            gyr = signal_filt[:, 3:]

            acc_mag = np.sqrt(np.sum(acc ** 2, axis=1, keepdims=True))
            gyr_mag = np.sqrt(np.sum(gyr ** 2, axis=1, keepdims=True))

            # Jerk (derivative of acceleration)
            jerk = np.gradient(acc, 1.0 / FS, axis=0)
            jerk_mag = np.sqrt(np.sum(jerk ** 2, axis=1, keepdims=True))

            # Combine: 6 original + acc_mag + gyr_mag + jerk_mag = 9 channels
            signal_full = np.hstack([signal_filt, acc_mag, gyr_mag, jerk_mag])

            # Z-score normalize per trial (ONCE only)
            signal_norm = zscore_normalize(signal_full)

            # Create windows
            w, l = create_windows(signal_norm, labels_raw, WINDOW_SAMPLES, TRAIN_OVERLAP, LABEL_THRESH)
            if len(w) > 0:
                all_windows.append(w)
                all_labels.append(l)

        if all_windows:
            subjects_data[sid] = {
                "windows": np.concatenate(all_windows, axis=0),
                "labels": np.concatenate(all_labels, axis=0),
            }
            n_fog = int(np.sum(subjects_data[sid]["labels"]))
            n_total = len(subjects_data[sid]["labels"])
            log.info("  Subject S%02d: %d windows (%d FoG, %d NoFoG)",
                     sid, n_total, n_fog, n_total - n_fog)

    return subjects_data

In [ ]:
print("=" * 90)
print("  FIGSHARE FoG DETECTION — IMPROVED PIPELINE")
print("=" * 90)
print(f"  Sampling rate: {FS} Hz")
print(f"  Window: {WINDOW_SEC}s ({WINDOW_SAMPLES} samples)")
print(f"  Bandpass: {BP_LOW}-{BP_HIGH} Hz")
print(f"  Label threshold: {LABEL_THRESH} (majority voting)")
print(f"  Feature selection: top {K_FEATURES} (mutual information)")
print(f"  Derived signals: acc_mag, gyr_mag, jerk_mag")
print(f"  Threshold optimisation: Youden's J")
print()

t0 = time.time()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: Load & preprocess
subjects_data = load_and_preprocess()

## Feature Extraction

In [ ]:
def extract_features_for_subject(windows: np.ndarray, fs: int = FS) -> pd.DataFrame:
    """Extract features from all windows of a subject."""
    extractor = FeatureExtractor(
        sampling_rate=fs,
        extract_time=True,
        extract_frequency=True,
        extract_wavelet=True,
        extract_nonlinear=False,
    )
    rows = []
    for i in range(len(windows)):
        feats = extractor.extract_from_window(windows[i], include_magnitude=False,
                                               channel_groups=None)
        rows.append(feats)
    return pd.DataFrame(rows)


def extract_all_features(subjects_data: Dict) -> Dict[int, Dict]:
    """Extract features for all subjects in parallel."""
    log.info("Extracting features for %d subjects (parallel)...", len(subjects_data))

    def _extract(sid):
        feat_df = extract_features_for_subject(subjects_data[sid]["windows"])
        return sid, feat_df

    results = Parallel(n_jobs=-1, verbose=0)(
        delayed(_extract)(sid) for sid in tqdm(subjects_data.keys(), desc="Feature extraction")
    )

    features = {}
    for sid, feat_df in results:
        features[sid] = {"X": feat_df, "y": subjects_data[sid]["labels"]}
        log.info("  S%02d: %d windows, %d features", sid, len(feat_df), feat_df.shape[1])
    return features

In [ ]:
# Step 2: Extract features
features = extract_all_features(subjects_data)